# Character-Level Text Generation — RNN vs GRU vs LSTM

This notebook builds three models on the same dataset:
- Simple RNN
- GRU
- LSTM

Goal:
- keep the code understandable for a lecture
- compare training behavior
- generate sample text from each model

How to read the code in this notebook:
- first we prepare the dataset and convert text into integer sequences
- then we define the model configuration and training setup
- finally we train three recurrent architectures under the same conditions and compare their outputs

The first code cell imports shared utilities from our local package. This keeps the notebook focused on the machine learning workflow instead of repeating all helper code inline.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "nirma_university_language_models").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [ ]:

import matplotlib.pyplot as plt

from nirma_university_language_models import (
    CharModel,
    build_dataloaders,
    build_vocabulary,
    encode_text,
    get_device,
    load_tinyshakespeare_text,
    sample_from_model,
    seed_everything,
    train_model,
)


In [ ]:
device = get_device()
print("Using device:", device)

SEED = 42
seed_everything(SEED)

## 1. Load Tiny Shakespeare

The setup code above chooses the device (`cpu` or `cuda`) and fixes the random seed. Setting the seed is important because it makes the experiment more reproducible: the same initialization and sampling choices are more likely to produce similar results across runs.

In the next code cell, we load the text, build the vocabulary, and encode the full corpus as integers. This gives us the raw material for training all three models.

In [ ]:
text, DATA_PATH = load_tinyshakespeare_text()
print("Dataset path:", DATA_PATH)

chars, char_to_idx, idx_to_char = build_vocabulary(text)
vocab_size = len(chars)
encoded = encode_text(text, char_to_idx)

print("Dataset length:", len(text))
print("Vocabulary size:", vocab_size)

## 2. Build dataset

Now we turn one long encoded text sequence into many supervised training examples.

What the next code cell does:
- `seq_len = 100` means each example contains 100 characters of context
- `build_dataloaders(...)` splits the corpus into training and validation parts
- PyTorch `DataLoader` objects handle batching so the model sees many examples per optimization step

Students should notice that the dataset length is much smaller than the raw character count because each item is a sequence window, not a single character.

In [ ]:
seq_len = 100
BATCH_SIZE = 64
train_loader, val_loader = build_dataloaders(encoded, seq_len=seq_len, batch_size=BATCH_SIZE)

print(len(train_loader.dataset), len(val_loader.dataset))

## 3. Define a configurable recurrent model

The actual class definition lives in the shared module, but this notebook still exposes the key hyperparameters.

In the next code cell:
- `embed_dim` controls the size of the vector used to represent each character
- `hidden_size` controls the size of the recurrent hidden state
- `preview_model` lets students inspect the layer structure before training starts

This is useful because students can see that changing `rnn_type` swaps the recurrent layer while leaving the rest of the pipeline unchanged.

In [ ]:
MODEL_KWARGS = {"embed_dim": 64, "hidden_size": 128}
preview_model = CharModel(vocab_size, rnn_type="RNN", **MODEL_KWARGS)
preview_model

## 4. Training and evaluation helpers

The heavy lifting now happens inside the shared `train_model(...)` helper.

Conceptually, that helper performs the usual deep learning loop:
- run the model forward on a batch
- compute cross-entropy loss against the correct next characters
- backpropagate the gradients
- update the model parameters with Adam
- evaluate on the validation set after each epoch

The next code cell prints the model configuration so students can connect the training output to the chosen hyperparameters.

In [ ]:
print("Training helper loaded from nirma_university_language_models.train_model")
print("Model config:", MODEL_KWARGS)

## 5. Train the three models

For a lecture, 3 to 5 epochs is usually enough to show the trend.

What the next code cell is doing:
- it trains an RNN, a GRU, and an LSTM one after another
- each model uses the same dataset, sequence length, embedding size, hidden size, learning rate, and number of epochs
- this makes the comparison fair, because the main difference is the recurrent architecture itself

Students should watch the printed training and validation losses after each epoch and look for which model learns faster or generalizes better.

In [ ]:
EPOCHS = 5

rnn_model, rnn_train_losses, rnn_val_losses = train_model(
    "RNN", vocab_size, train_loader, val_loader, device, epochs=EPOCHS, **MODEL_KWARGS
)
gru_model, gru_train_losses, gru_val_losses = train_model(
    "GRU", vocab_size, train_loader, val_loader, device, epochs=EPOCHS, **MODEL_KWARGS
)
lstm_model, lstm_train_losses, lstm_val_losses = train_model(
    "LSTM", vocab_size, train_loader, val_loader, device, epochs=EPOCHS, **MODEL_KWARGS
)

## 6. Compare loss curves

Numbers printed during training are useful, but plots make trends easier to compare.

The next code cell draws six lines:
- training loss for RNN, GRU, and LSTM
- validation loss for RNN, GRU, and LSTM

When students read the graph, they should ask two questions:
- which model reaches a lower loss?
- is validation loss improving in the same direction as training loss?

In [ ]:
epochs = list(range(1, EPOCHS + 1))

plt.figure(figsize=(10, 5))
plt.plot(epochs, rnn_train_losses, marker="o", label="RNN Train")
plt.plot(epochs, rnn_val_losses, marker="o", label="RNN Val")
plt.plot(epochs, gru_train_losses, marker="o", label="GRU Train")
plt.plot(epochs, gru_val_losses, marker="o", label="GRU Val")
plt.plot(epochs, lstm_train_losses, marker="o", label="LSTM Train")
plt.plot(epochs, lstm_val_losses, marker="o", label="LSTM Val")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.tight_layout()
plt.show()

## 7. Generate text from a trained model

Loss tells us how well the model predicts the next character on average, but generated text gives a more intuitive sense of what the model has learned.

In this section:
- `start_text` is the prompt we feed into the model first
- `length` is how many new characters we ask it to generate
- `temperature` controls randomness; lower values are safer and more repetitive, higher values are more diverse and risky

The following code cells sample one output from each trained model so students can compare fluency, repetition, and structure.

In [ ]:
SAMPLE_KWARGS = {"start_text": "ROMEO:", "length": 400, "temperature": 0.8}
print("Sampling config:", SAMPLE_KWARGS)

In [ ]:
print("=== RNN Sample ===")
print(sample_from_model(rnn_model, char_to_idx, idx_to_char, device, **SAMPLE_KWARGS))

In [ ]:
print("=== GRU Sample ===")
print(sample_from_model(gru_model, char_to_idx, idx_to_char, device, **SAMPLE_KWARGS))

In [ ]:
print("=== LSTM Sample ===")
print(sample_from_model(lstm_model, char_to_idx, idx_to_char, device, **SAMPLE_KWARGS))